# LeetCode 387 — First Unique Character in a String bằng Python

**Link đề:** https://leetcode.com/problems/first-unique-character-in-a-string/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String, Queue, Counting

```python
class Solution:
    def firstUniqChar(self, s: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu "ký tự **không lặp** đầu tiên" và trả về **chỉ số** (không phải ký tự).
2. Đi từ cách so mọi cặp `O(n²)` → **đếm tần suất hai lượt (two-pass counting)** `O(n)`.
3. Biết các cách khác: `find`/`rfind` cho 26 chữ cái, lưu **chỉ số** trong một lượt, **hàng đợi** cho dữ liệu dạng luồng.
4. Trace bảng đếm và lượt duyệt thứ hai; chứng minh vì sao phải duyệt **theo thứ tự chuỗi**, không theo thứ tự dict.
5. Nhận dạng pattern "**đếm trước, quyết định sau**".

## 1. Đề bài

Cho chuỗi `s`. Tìm ký tự **không lặp lại** (chỉ xuất hiện đúng 1 lần) **đầu tiên** và trả về **chỉ số** của nó. Nếu không có, trả `-1`.

| `s` | Output | Vì sao |
|---|---:|---|
| `"leetcode"` | `0` | `'l'` chỉ xuất hiện 1 lần, ở chỉ số 0 |
| `"loveleetcode"` | `2` | `l`, `o` lặp; `'v'` ở chỉ số 2 là ký tự duy nhất đầu tiên |
| `"aabb"` | `-1` | mọi ký tự đều lặp |

**Constraints:** `1 <= len(s) <= 10^5`; `s` chỉ gồm chữ cái tiếng Anh thường.

**Lưu ý:** "đầu tiên" là theo **vị trí trong chuỗi**, không phải theo bảng chữ cái.

In [1]:
s = "loveleetcode"
for i, ch in enumerate(s):
    print(i, ch, "xuất hiện", s.count(ch), "lần")

0 l xuất hiện 2 lần
1 o xuất hiện 2 lần
2 v xuất hiện 1 lần
3 e xuất hiện 4 lần
4 l xuất hiện 2 lần
5 e xuất hiện 4 lần
6 e xuất hiện 4 lần
7 t xuất hiện 1 lần
8 c xuất hiện 1 lần
9 o xuất hiện 2 lần
10 d xuất hiện 1 lần
11 e xuất hiện 4 lần


Ở trên ta thấy chỉ số `2` (`v`) là vị trí **đầu tiên** có số lần = 1. Các ký tự `t` (7), `c` (8), `d` (10) cũng chỉ xuất hiện 1 lần nhưng đứng **sau** nên không được chọn.

## 2. Nền tảng Python

### 2.1. `enumerate` — vừa lấy chỉ số vừa lấy ký tự

`for i, ch in enumerate(s):` cho từng cặp `(chỉ_số, ký_tự)`. Đề yêu cầu **chỉ số**, nên `enumerate` là công cụ tự nhiên.

In [2]:
for i, ch in enumerate("abc"):
    print(f"i={i}, ch={ch!r}")
print(list(enumerate("abc")))

i=0, ch='a'
i=1, ch='b'
i=2, ch='c'
[(0, 'a'), (1, 'b'), (2, 'c')]


### 2.2. `dict` và `Counter` để đếm

`cnt[ch] = cnt.get(ch, 0) + 1` hoặc `Counter(s)`. Từ Python 3.7, `dict` **giữ thứ tự chèn** (thứ tự lần đầu gặp khóa) — chi tiết này sẽ dùng ở mục 7.

In [3]:
from collections import Counter
cnt = {}
for ch in "leetcode":
    cnt[ch] = cnt.get(ch, 0) + 1
print(cnt)
print(Counter("leetcode"))
print("Thứ tự khóa của dict = thứ tự gặp lần đầu:", list(cnt))

{'l': 1, 'e': 3, 't': 1, 'c': 1, 'o': 1, 'd': 1}
Counter({'e': 3, 'l': 1, 't': 1, 'c': 1, 'o': 1, 'd': 1})
Thứ tự khóa của dict = thứ tự gặp lần đầu: ['l', 'e', 't', 'c', 'o', 'd']


### 2.3. `str.find` và `str.rfind`

- `s.find(ch)` — chỉ số **đầu tiên** của `ch` (hoặc `-1`).
- `s.rfind(ch)` — chỉ số **cuối cùng** của `ch` (hoặc `-1`).

Ký tự xuất hiện **đúng 1 lần** ⇔ `find == rfind != -1`.

In [4]:
s = "loveleetcode"
for ch in "lvez":
    print(ch, "find =", s.find(ch), "| rfind =", s.rfind(ch), "| duy nhất?", s.find(ch) == s.rfind(ch) != -1)

l find = 0 | rfind = 4 | duy nhất? False
v find = 2 | rfind = 2 | duy nhất? True
e find = 3 | rfind = 11 | duy nhất? False
z find = -1 | rfind = -1 | duy nhất? False


`a == b != -1` là **so sánh nối chuỗi** của Python, nghĩa là `(a == b) and (b != -1)`.

## 3. Cách 1 — Trực giác: với mỗi vị trí, kiểm tra có ký tự nào khác bằng nó

Duyệt `i` từ trái sang phải; với mỗi `i`, quét toàn bộ chuỗi tìm `j != i` có `s[j] == s[i]`. Không tìm thấy → `i` là đáp án.

In [5]:
def first_uniq_brute(s: str) -> int:
    n = len(s)
    for i in range(n):
        repeated = False
        for j in range(n):
            if j != i and s[j] == s[i]:
                repeated = True
                break
        if not repeated:
            return i
    return -1

for s in ["leetcode", "loveleetcode", "aabb"]:
    print(s, "->", first_uniq_brute(s))

leetcode -> 0
loveleetcode -> 2
aabb -> -1


Cách "ngắn hơn" `s.count(s[i]) == 1` cũng là `O(n)` cho mỗi `i` → tổng vẫn `O(n²)`.

### 3.1. Nút thắt

Với `n = 10^5`, `O(n²) = 10^10` → quá chậm. Ta lặp đi lặp lại câu hỏi "`s[i]` xuất hiện mấy lần?" — mỗi lần lại đếm lại từ đầu.

**Insight:** **đếm một lần** cho cả chuỗi (`O(n)`), lưu vào bảng; sau đó câu hỏi "xuất hiện mấy lần" chỉ là tra bảng `O(1)`. Rồi duyệt **lại chuỗi theo thứ tự** để tìm vị trí đầu tiên có số đếm = 1.

In [6]:
def first_uniq_count_method(s: str) -> int:
    for i, ch in enumerate(s):
        if s.count(ch) == 1:          # O(n) mỗi lần
            return i
    return -1

import time
worst = "ab" * 10000 + "c"            # ký tự duy nhất ở tận cuối -> phải gọi count cho mọi vị trí
t0 = time.perf_counter(); r = first_uniq_count_method(worst); t1 = time.perf_counter()
print(f"n={len(worst)}: kết quả {r}, s.count cho từng vị trí mất ~{t1 - t0:.2f}s")
print("Số ký tự phải quét ≈ n * n =", f"{len(worst) ** 2:,}", "| với n = 10^5 sẽ là", f"{(10**5) ** 2:,}")

n=20001: kết quả 20000, s.count cho từng vị trí mất ~0.21s
Số ký tự phải quét ≈ n * n = 400,040,001 | với n = 10^5 sẽ là 10,000,000,000


## 4. Cách 2 — Đếm hai lượt (lời giải chính)

**Lượt 1 — đếm:** `cnt[ch]` = số lần `ch` xuất hiện trong `s`.

**Lượt 2 — tìm:** duyệt lại `s` **theo thứ tự chỉ số**; vị trí `i` đầu tiên có `cnt[s[i]] == 1` là đáp án.

**Không tìm thấy** → `-1`.

In [7]:
def first_uniq_two_pass(s: str) -> int:
    cnt = {}
    for ch in s:                       # Lượt 1
        cnt[ch] = cnt.get(ch, 0) + 1
    for i, ch in enumerate(s):         # Lượt 2 — theo thứ tự chuỗi
        if cnt[ch] == 1:
            return i
    return -1

for s in ["leetcode", "loveleetcode", "aabb"]:
    print(s, "->", first_uniq_two_pass(s))

leetcode -> 0
loveleetcode -> 2
aabb -> -1


## 5. TRACE — `s = "loveleetcode"`

**Lượt 1** cho `cnt = {l: 2, o: 2, v: 1, e: 4, t: 1, c: 1, d: 1}`.

**Lượt 2:**

| `i` | `s[i]` | `cnt[s[i]]` | Quyết định |
|---:|:---:|---:|---|
| 0 | l | 2 | lặp → đi tiếp |
| 1 | o | 2 | lặp → đi tiếp |
| 2 | v | 1 | **duy nhất → trả 2** |

Chú ý `t`, `c`, `d` cũng có số đếm 1 nhưng **đứng sau** `v` nên không được chọn.

In [8]:
def trace_first_uniq(s: str) -> int:
    cnt = {}
    for ch in s:
        cnt[ch] = cnt.get(ch, 0) + 1
    print(f"s={s!r}\nLượt 1 -> cnt = {cnt}")
    for i, ch in enumerate(s):
        if cnt[ch] == 1:
            print(f"  i={i}, ch={ch!r}, cnt={cnt[ch]} -> DUY NHẤT, trả {i}")
            return i
        print(f"  i={i}, ch={ch!r}, cnt={cnt[ch]} -> lặp, đi tiếp")
    print("  Hết chuỗi -> -1")
    return -1

_ = trace_first_uniq("loveleetcode")
_ = trace_first_uniq("aabb")

s='loveleetcode'
Lượt 1 -> cnt = {'l': 2, 'o': 2, 'v': 1, 'e': 4, 't': 1, 'c': 1, 'd': 1}
  i=0, ch='l', cnt=2 -> lặp, đi tiếp
  i=1, ch='o', cnt=2 -> lặp, đi tiếp
  i=2, ch='v', cnt=1 -> DUY NHẤT, trả 2
s='aabb'
Lượt 1 -> cnt = {'a': 2, 'b': 2}
  i=0, ch='a', cnt=2 -> lặp, đi tiếp
  i=1, ch='a', cnt=2 -> lặp, đi tiếp
  i=2, ch='b', cnt=2 -> lặp, đi tiếp
  i=3, ch='b', cnt=2 -> lặp, đi tiếp
  Hết chuỗi -> -1


## 6. Cách 3 — Mảng 26 phần tử

Chỉ có 26 chữ thường → thay `dict` bằng list 26 số đếm, chỉ số `ord(ch) - ord('a')` (`ord('a') = 97`).

In [9]:
def first_uniq_array(s: str) -> int:
    counts = [0] * 26
    for ch in s:
        counts[ord(ch) - 97] += 1
    for i, ch in enumerate(s):
        if counts[ord(ch) - 97] == 1:
            return i
    return -1

print([first_uniq_array(s) for s in ["leetcode", "loveleetcode", "aabb"]])

[0, 2, -1]


## 7. Cách 4 — Duyệt bảng đếm thay vì duyệt lại chuỗi

Khi chuỗi **rất dài** nhưng bảng chữ cái nhỏ, lượt 2 có thể duyệt **bảng** (≤ 26 khóa) thay vì cả chuỗi. Ta lưu **chỉ số lần đầu** của mỗi ký tự, và đánh dấu ký tự lặp bằng giá trị đặc biệt (ví dụ `-1`). Cuối cùng lấy **chỉ số nhỏ nhất** trong các ký tự không lặp.

`min(iterable, default=-1)` trả phần tử nhỏ nhất, hoặc `-1` nếu rỗng.

In [10]:
def first_uniq_index_map(s: str) -> int:
    first = {}                         # ch -> chỉ số lần đầu, hoặc -1 nếu đã lặp
    for i, ch in enumerate(s):
        if ch in first:
            first[ch] = -1             # đã thấy trước đó -> lặp
        else:
            first[ch] = i
    return min((i for i in first.values() if i != -1), default=-1)

print([first_uniq_index_map(s) for s in ["leetcode", "loveleetcode", "aabb"]])
print("min([], default=-1) =", min([], default=-1))

[0, 2, -1]
min([], default=-1) = -1


## 8. Cách 5 — `find` / `rfind` cho từng chữ cái

Với mỗi chữ `c` trong `'a'..'z'`: nếu `s.find(c) == s.rfind(c) != -1` thì `c` duy nhất ở vị trí `s.find(c)`. Lấy `min` các vị trí đó. Chỉ 26 lần, mỗi lần quét `O(n)` (viết bằng C nên rất nhanh) → `O(26·n)`.

`string.ascii_lowercase` là hằng `'abcdefghijklmnopqrstuvwxyz'` trong thư viện chuẩn.

In [11]:
import string

def first_uniq_find(s: str) -> int:
    best = len(s)
    for c in string.ascii_lowercase:
        i = s.find(c)
        if i != -1 and i == s.rfind(c):
            best = min(best, i)
    return best if best < len(s) else -1

print(string.ascii_lowercase)
print([first_uniq_find(s) for s in ["leetcode", "loveleetcode", "aabb"]])

abcdefghijklmnopqrstuvwxyz
[0, 2, -1]


`best if best < len(s) else -1` là **biểu thức điều kiện** (ternary): giá trị `best` nếu điều kiện đúng, ngược lại `-1`.

## 9. Cách 6 — Hàng đợi cho dữ liệu dạng luồng (stream)

Nếu ký tự đến **từng cái một** và ta muốn biết "ký tự duy nhất đầu tiên **tính tới lúc này**" sau mỗi ký tự: giữ một **hàng đợi** các `(ký_tự, chỉ_số)` ứng viên; sau mỗi ký tự mới, **bỏ khỏi đầu hàng** các ứng viên đã bị lặp. Đầu hàng luôn là đáp án hiện tại. Mỗi phần tử vào/ra hàng tối đa 1 lần → tổng `O(n)`.

In [12]:
from collections import deque

def first_uniq_stream(s: str) -> list[int]:
    cnt = {}
    q = deque()
    answers = []
    for i, ch in enumerate(s):
        cnt[ch] = cnt.get(ch, 0) + 1
        if cnt[ch] == 1:
            q.append((ch, i))
        while q and cnt[q[0][0]] > 1:     # đầu hàng đã bị lặp -> loại
            q.popleft()
        answers.append(q[0][1] if q else -1)
    return answers

s = "loveleetcode"
print(list(s))
print(first_uniq_stream(s))
print("Đáp án của bài (cuối luồng):", first_uniq_stream(s)[-1])

['l', 'o', 'v', 'e', 'l', 'e', 'e', 't', 'c', 'o', 'd', 'e']
[0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2]
Đáp án của bài (cuối luồng): 2


Đọc kết quả: chỉ số 0–3 đáp án là `0` (`l` vẫn duy nhất). Tại chỉ số 4, `l` lặp → bị loại khỏi đầu hàng → đáp án thành `1` (`o`). Tại chỉ số 9, `o` lặp → bị loại → đáp án thành `2` (`v`) và giữ tới cuối. Phần tử cuối của list chính là đáp án của bài.

## 10. Code nộp LeetCode

In [13]:
class Solution:
    def firstUniqChar(self, s: str) -> int:
        cnt = {}
        for ch in s:
            cnt[ch] = cnt.get(ch, 0) + 1
        for i, ch in enumerate(s):
            if cnt[ch] == 1:
                return i
        return -1


sol = Solution()
print([sol.firstUniqChar(s) for s in ["leetcode", "loveleetcode", "aabb"]])

[0, 2, -1]


Bản nộp ngắn bằng `Counter`:

```python
from collections import Counter

class Solution:
    def firstUniqChar(self, s: str) -> int:
        cnt = Counter(s)
        for i, ch in enumerate(s):
            if cnt[ch] == 1:
                return i
        return -1
```

## 11. Vì sao đúng? + Độ phức tạp

**Đúng:** sau lượt 1, `cnt[ch]` đúng bằng số lần xuất hiện của `ch` trong **toàn bộ** `s` (không chỉ phần đã duyệt). Lượt 2 xét các chỉ số theo thứ tự tăng dần `0, 1, 2, ...`; chỉ số đầu tiên thỏa `cnt[s[i]] == 1` chính là chỉ số **nhỏ nhất** của một ký tự duy nhất — đúng định nghĩa "đầu tiên". Nếu không chỉ số nào thỏa thì không có ký tự duy nhất → `-1`.

**Vì sao không thể trả lời ngay trong lượt 1?** Khi đang ở vị trí `i`, ta chưa biết `s[i]` có lặp **ở phía sau** hay không. Phải đếm hết mới quyết định được — đây là lý do cần **hai lượt** (hoặc bảng chỉ số như Cách 4).

Gọi `n = len(s)`, `Σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. So mọi cặp / `s.count` | `O(n²)` | `O(1)` |
| 2. Đếm hai lượt (dict) | `O(n)` | `O(Σ)` |
| 3. Mảng 26 | `O(n)` | `O(Σ)` = `O(1)` |
| 4. Bảng chỉ số lần đầu | `O(n + Σ)` | `O(Σ)` |
| 5. `find`/`rfind` 26 chữ | `O(Σ·n)` | `O(1)` |
| 6. Hàng đợi (stream) | `O(n)` tổng, `O(1)` phân bổ mỗi ký tự | `O(Σ)` |

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [14]:
methods = {
    "brute": first_uniq_brute,
    "count_method": first_uniq_count_method,
    "two_pass": first_uniq_two_pass,
    "array": first_uniq_array,
    "index_map": first_uniq_index_map,
    "find": first_uniq_find,
    "stream_last": lambda s: first_uniq_stream(s)[-1],
    "Solution": Solution().firstUniqChar,
}
cases = [
    ("leetcode", 0), ("loveleetcode", 2), ("aabb", -1),     # mẫu đề
    ("z", 0),                                               # 1 ký tự
    ("zz", -1),
    ("aab", 2),                                             # duy nhất ở cuối
    ("abcabcd", 6),
    ("dddccdbba", 8),
    ("abcdefghijklmnopqrstuvwxyz", 0),
]
for s, exp in cases:
    for name, f in methods.items():
        assert f(s) == exp, (name, s, f(s), exp)
    print(f"{s!r:<30} -> {exp:>2} PASS")

'leetcode'                     ->  0 PASS
'loveleetcode'                 ->  2 PASS
'aabb'                         -> -1 PASS
'z'                            ->  0 PASS
'zz'                           -> -1 PASS
'aab'                          ->  2 PASS
'abcabcd'                      ->  6 PASS
'dddccdbba'                    ->  8 PASS
'abcdefghijklmnopqrstuvwxyz'   ->  0 PASS


In [15]:
import random
rng = random.Random(387)
for _ in range(1000):
    s = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 15)))
    exp = first_uniq_brute(s)
    for name, f in methods.items():
        assert f(s) == exp, (name, s)
big = "".join(rng.choice(string.ascii_lowercase) for _ in range(10**5))
assert Solution().firstUniqChar(big) == first_uniq_find(big)
print("1000 test ngẫu nhiên + 1 chuỗi 10^5 ký tự PASS")

1000 test ngẫu nhiên + 1 chuỗi 10^5 ký tự PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Trả về ký tự thay vì chỉ số
`return ch` → `'l'` thay vì `0`. Đề yêu cầu **index**.

### Lỗi 2 — Lượt 2 duyệt theo bảng đếm, không theo chuỗi (và không lưu chỉ số)
`for ch in cnt: if cnt[ch] == 1: return s.index(ch)` — trông đúng (vì dict giữ thứ tự gặp lần đầu), nhưng nếu bảng đếm được dựng từ `Counter(s).most_common()` hoặc `sorted(cnt)` thì thứ tự sai. Và nếu dùng mảng 26, duyệt `0..25` là thứ tự **bảng chữ cái** → sai.

### Lỗi 3 — Quyết định ngay trong lượt 1
"Gặp ký tự lần đầu thì coi như duy nhất" — sai vì có thể lặp ở phía sau.

### Lỗi 4 — Quên trường hợp không có → trả `None` thay vì `-1`
Hàm Python không có `return` ở cuối sẽ trả `None`.

In [16]:
def wrong_return_char(s):
    cnt = Counter(s)
    for ch in s:
        if cnt[ch] == 1:
            return ch

def wrong_alphabet_order(s):
    counts = [0] * 26
    for ch in s:
        counts[ord(ch) - 97] += 1
    for k in range(26):                 # duyệt theo bảng chữ cái — SAI
        if counts[k] == 1:
            return s.index(chr(k + 97))  # chr: ngược lại của ord
    return -1

def wrong_one_pass(s):
    seen = set()
    for i, ch in enumerate(s):
        if ch not in seen:
            return i                    # vội kết luận
        seen.add(ch)
    return -1

def wrong_no_minus_one(s):
    cnt = Counter(s)
    for i, ch in enumerate(s):
        if cnt[ch] == 1:
            return i

print("Lỗi 1:", wrong_return_char("leetcode"), "(đúng: 0)")
print("Lỗi 2:", wrong_alphabet_order("zbba"), "(đúng: 0 vì 'z' duy nhất và đứng trước 'a')")
print("Lỗi 3:", wrong_one_pass("aab"), "(đúng: 2)")
print("Lỗi 4:", wrong_no_minus_one("aabb"), "(đúng: -1)")
print("chr(97) =", chr(97), "| ord('a') =", ord("a"))

Lỗi 1: l (đúng: 0)
Lỗi 2: 3 (đúng: 0 vì 'z' duy nhất và đứng trước 'a')
Lỗi 3: 0 (đúng: 2)
Lỗi 4: None (đúng: -1)
chr(97) = a | ord('a') = 97


## 14. Mẹo nhận dạng pattern

- "Đầu tiên/cuối cùng **thỏa một tính chất phụ thuộc toàn bộ dữ liệu**" (duy nhất, xuất hiện k lần...) → **đếm trước (lượt 1), quyết định sau (lượt 2)**.
- Trả **chỉ số** → lượt 2 phải duyệt **theo thứ tự chỉ số** của dữ liệu gốc.
- Dữ liệu dạng luồng, cần trả lời liên tục → **hàng đợi + bộ đếm** (loại lười ở đầu hàng).
- Bài anh em: **451 Sort Characters By Frequency**, **1636**, **2351 First Letter to Appear Twice**, **1429 First Unique Number** (thiết kế, dạng luồng).

## 15. Biến thể

### 15.1. Ký tự đầu tiên xuất hiện **hai lần** (LeetCode 2351)

Ngược lại: ở đây quyết định **được ngay** trong một lượt — lần thứ hai gặp chữ nào thì trả chữ đó.

In [17]:
def repeated_character(s: str) -> str:
    seen = set()
    for ch in s:
        if ch in seen:
            return ch
        seen.add(ch)
    return ""

print(repeated_character("abccbaacz"), repeated_character("abcdd"))

c d


### 15.2. Chỉ số của ký tự xuất hiện **đúng k lần** đầu tiên

Tổng quát hóa điều kiện `== 1` thành `== k`.

In [18]:
def first_with_count(s: str, k: int) -> int:
    cnt = Counter(s)
    for i, ch in enumerate(s):
        if cnt[ch] == k:
            return i
    return -1

print(first_with_count("loveleetcode", 2), first_with_count("loveleetcode", 4), first_with_count("abc", 3))

0 3 -1


## 16. Bài tập tự luyện (TODO)

**Bài 1 (tính tay).** Lập bảng `cnt` và lượt 2 cho `s = "dddccdbba"`. Đáp án?

**Bài 2 (dự đoán output).** `first_uniq_stream("aabcbc")` trả list gì?

**Bài 3 (điền chỗ trống).** Hoàn thiện `exercise_two_pass` dùng mảng 26.

**Bài 4 (viết lại).** Trả về **ký tự** duy nhất cuối cùng (theo vị trí); không có thì `""`.

**Bài 5 (biến thể).** Trả về **danh sách chỉ số** của mọi ký tự duy nhất, theo thứ tự tăng.

**Bài 6 (mini challenge — First Unique Number, LeetCode 1429).** Thiết kế class `FirstUnique(nums)` với `showFirstUnique() -> int` (trả `-1` nếu không có) và `add(value)`; mỗi thao tác `O(1)` phân bổ.

In [19]:
def exercise_two_pass(s: str) -> int:
    # TODO 1: counts = [0] * 26, đếm
    # TODO 2: duyệt enumerate(s), trả i đầu tiên có counts == 1
    # TODO 3: return -1
    pass

# assert exercise_two_pass("loveleetcode") == 2 and exercise_two_pass("aabb") == -1

In [20]:
def last_unique_char(s: str) -> str:
    # TODO
    pass

# assert last_unique_char("loveleetcode") == "d" and last_unique_char("aabb") == ""

In [21]:
def all_unique_indices(s: str) -> list[int]:
    # TODO
    pass

# assert all_unique_indices("loveleetcode") == [2, 7, 8, 10]

In [22]:
class FirstUnique:
    def __init__(self, nums: list[int]):
        # TODO
        pass

    def showFirstUnique(self) -> int:
        # TODO
        pass

    def add(self, value: int) -> None:
        # TODO
        pass

# fu = FirstUnique([2, 3, 5]); assert fu.showFirstUnique() == 2
# fu.add(5); fu.add(2); assert fu.showFirstUnique() == 3; fu.add(3); assert fu.showFirstUnique() == -1

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đếm d, c, b, a | `{d: 4, c: 2, b: 2, a: 1}` | chỉ `a` có 1 → vị trí của nó |
| 2 | giữ hàng đợi ứng viên | lặp thì loại khỏi đầu hàng | ghi đáp án sau **mỗi** ký tự |
| 3 | giống Cách 3 | `ord(ch) - 97` | `if counts[...] == 1: return i` |
| 4 | duyệt ngược | `reversed(s)` hoặc `range(len(s) - 1, -1, -1)` | trả ký tự đầu tiên có đếm 1 khi đi ngược |
| 5 | lượt 2 không dừng sớm | `append(i)` | list comprehension có `if` |
| 6 | giống Cách 6 | `Counter` + `deque` | `showFirstUnique` loại đầu hàng đã lặp |

## 18. Đáp án tham khảo + Template + Checklist

**Bài 1:** `cnt = {d: 4, c: 2, b: 2, a: 1}` → duyệt: d, d, d, c, c, d, b, b đều lặp → `a` ở chỉ số **8**.

In [23]:
def answer_two_pass(s):
    counts = [0] * 26
    for ch in s:
        counts[ord(ch) - 97] += 1
    for i, ch in enumerate(s):
        if counts[ord(ch) - 97] == 1:
            return i
    return -1

def answer_last_unique_char(s):
    cnt = Counter(s)
    for ch in reversed(s):
        if cnt[ch] == 1:
            return ch
    return ""

def answer_all_unique_indices(s):
    cnt = Counter(s)
    return [i for i, ch in enumerate(s) if cnt[ch] == 1]

class AnswerFirstUnique:
    def __init__(self, nums):
        self.cnt = Counter()
        self.q = deque()
        for x in nums:
            self.add(x)
    def showFirstUnique(self):
        while self.q and self.cnt[self.q[0]] > 1:
            self.q.popleft()
        return self.q[0] if self.q else -1
    def add(self, value):
        self.cnt[value] += 1
        if self.cnt[value] == 1:
            self.q.append(value)

_ = trace_first_uniq("dddccdbba")
print("Bài 2:", first_uniq_stream("aabcbc"))
assert first_uniq_stream("aabcbc") == [0, -1, 2, 2, 3, -1]
print("Bài 5 (loveleetcode):", answer_all_unique_indices("loveleetcode"))
assert answer_two_pass("loveleetcode") == 2 and answer_two_pass("aabb") == -1
assert answer_last_unique_char("loveleetcode") == "d" and answer_last_unique_char("aabb") == ""
assert answer_all_unique_indices("loveleetcode") == [2, 7, 8, 10]
fu = AnswerFirstUnique([2, 3, 5]); assert fu.showFirstUnique() == 2
fu.add(5); fu.add(2); assert fu.showFirstUnique() == 3
fu.add(3); assert fu.showFirstUnique() == -1
print("Đáp án PASS toàn bộ!")

s='dddccdbba'
Lượt 1 -> cnt = {'d': 4, 'c': 2, 'b': 2, 'a': 1}
  i=0, ch='d', cnt=4 -> lặp, đi tiếp
  i=1, ch='d', cnt=4 -> lặp, đi tiếp
  i=2, ch='d', cnt=4 -> lặp, đi tiếp
  i=3, ch='c', cnt=2 -> lặp, đi tiếp
  i=4, ch='c', cnt=2 -> lặp, đi tiếp
  i=5, ch='d', cnt=4 -> lặp, đi tiếp
  i=6, ch='b', cnt=2 -> lặp, đi tiếp
  i=7, ch='b', cnt=2 -> lặp, đi tiếp
  i=8, ch='a', cnt=1 -> DUY NHẤT, trả 8
Bài 2: [0, -1, 2, 2, 3, -1]
Bài 5 (loveleetcode): [2, 7, 8, 10]
Đáp án PASS toàn bộ!


**Bài 2:** `[0, -1, 2, 2, 3, -1]` — chỉ số 0: `a` duy nhất; 1: `a` lặp, hàng rỗng → `-1`; 2: `b` → 2; 3: `c` thêm vào cuối hàng, đầu vẫn `b` → 2; 4: `b` lặp → đầu hàng thành `c` → 3; 5: `c` lặp → rỗng → `-1`. (Output thật ở cell trên.)

**Bài 5:** `[2, 7, 8, 10]` (`v`, `t`, `c`, `d`).

### Template nhớ nhanh

```python
class Solution:
    def firstUniqChar(self, s: str) -> int:
        cnt = {}
        for ch in s:                     # lượt 1: đếm TOÀN BỘ
            cnt[ch] = cnt.get(ch, 0) + 1
        for i, ch in enumerate(s):       # lượt 2: theo thứ tự chỉ số
            if cnt[ch] == 1:
                return i
        return -1
```

### Checklist submit
- [ ] Trả **chỉ số** (int), không phải ký tự?
- [ ] Lượt 2 duyệt theo **thứ tự chuỗi**?
- [ ] Có `return -1` cuối hàm?
- [ ] Đã thử: duy nhất ở đầu, ở cuối, không có, chuỗi 1 ký tự?